# US 4.4 -- Domain Confusion Analysis

After training DANN, we need to verify that the encoder has actually learned
domain-invariant features.  This notebook shows how to:

1. Extract bottleneck features from a trained model
2. Compute and plot t-SNE embeddings (before vs. after DANN)
3. Measure domain classifier accuracy (the "domain confusion score")

## CLI equivalent

```bash
udm-epic4 analyze --checkpoint outputs/epic4_dann/best.pth \
                   --config configs/epic4_dann.yaml
```

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from udm_epic4.models.dann import DANNModel
from udm_epic4.evaluation.domain_analysis import (
    extract_features,
    compute_tsne,
    plot_tsne,
    domain_confusion_score,
)

---
## 1. Feature Extraction

The `extract_features` function:
1. Runs images through the encoder.
2. Takes the **bottleneck** (deepest) feature map.
3. Global-average-pools it to a 1-D vector per sample.

The result is a `[N, feat_dim]` array (e.g. `[N, 768]` for ConvNeXt-Tiny).

In [ ]:
# Demo: extract features from a model on random data
model = DANNModel(backbone="convnext_tiny", pretrained=False)
model.eval()

# The encode() method extracts bottleneck features
dummy = torch.randn(4, 3, 512, 512)
with torch.no_grad():
    bottleneck = model.encode(dummy)
    print(f"Bottleneck shape: {bottleneck.shape}")  # [4, 768, 16, 16]

    # Global average pool to get feature vectors
    pooled = torch.nn.AdaptiveAvgPool2d(1)(bottleneck).flatten(1)
    print(f"Pooled features:  {pooled.shape}")      # [4, 768]

### Using `extract_features` with real data

In practice, you pass a `DomainDataset` and the function handles batching:

```python
from udm_epic4.data.multi_domain_dataset import DomainDataset

source_ds = DomainDataset('data/warstein/images', 'data/warstein/masks', domain_label=0)
target_ds = DomainDataset('data/malaysia/images', domain_label=1)

src_feats, src_labels = extract_features(model, source_ds, device='cuda', max_samples=500)
tgt_feats, tgt_labels = extract_features(model, target_ds, device='cuda', max_samples=500)

# Combine for t-SNE
all_feats = np.concatenate([src_feats, tgt_feats])
all_labels = np.concatenate([src_labels, tgt_labels])
```

---
## 2. t-SNE Visualisation

t-SNE reduces the high-dimensional feature vectors to 2D for visualisation.
We expect:

- **Before DANN (baseline):** Source and target form separate clusters.
- **After DANN:** Source and target are interleaved (domain-invariant features).

In [ ]:
# Simulated t-SNE comparison (replace with real features when available)
np.random.seed(42)

# Before DANN: two separate clusters
n_per_domain = 200
before_source = np.random.randn(n_per_domain, 2) * 0.8 + np.array([-2, 0])
before_target = np.random.randn(n_per_domain, 2) * 0.8 + np.array([2, 0])
before_coords = np.vstack([before_source, before_target])
before_labels = np.array([0] * n_per_domain + [1] * n_per_domain)

# After DANN: overlapping clusters
after_source = np.random.randn(n_per_domain, 2) * 1.2 + np.array([0, 0])
after_target = np.random.randn(n_per_domain, 2) * 1.2 + np.array([0.3, 0.2])
after_coords = np.vstack([after_source, after_target])
after_labels = np.array([0] * n_per_domain + [1] * n_per_domain)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before DANN
for lbl, name, color in [(0, 'Source (warstein)', '#2196F3'), (1, 'Target (malaysia)', '#FF5722')]:
    mask = before_labels == lbl
    axes[0].scatter(before_coords[mask, 0], before_coords[mask, 1],
                    c=color, label=name, alpha=0.5, s=15, edgecolors='none')
axes[0].set_title('Before DANN (Baseline)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_xlabel('t-SNE dim 1')
axes[0].set_ylabel('t-SNE dim 2')
axes[0].grid(True, alpha=0.2)

# After DANN
for lbl, name, color in [(0, 'Source (warstein)', '#2196F3'), (1, 'Target (malaysia)', '#FF5722')]:
    mask = after_labels == lbl
    axes[1].scatter(after_coords[mask, 0], after_coords[mask, 1],
                    c=color, label=name, alpha=0.5, s=15, edgecolors='none')
axes[1].set_title('After DANN', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].set_xlabel('t-SNE dim 1')
axes[1].set_ylabel('t-SNE dim 2')
axes[1].grid(True, alpha=0.2)

fig.suptitle('t-SNE of Encoder Bottleneck Features (Simulated)', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print("Left: domains form separate clusters (baseline) -- domain-specific features.")
print("Right: domains overlap (DANN) -- domain-invariant features.")

### Running t-SNE on real features

```python
# With real data and a trained model:
coords = compute_tsne(all_feats, perplexity=30.0, seed=42)
plot_tsne(coords, all_labels, domain_names=['warstein', 'malaysia'],
          save_path='outputs/epic4_dann/analysis/tsne_domains.png')
```

The `compute_tsne` function uses sklearn's t-SNE with PCA initialisation
for reproducible results.

---
## 3. Domain Confusion Score

The domain confusion score trains a small linear probe on the frozen encoder
features to classify source vs. target.  The probe's accuracy tells us how
separable the domains are:

| Probe accuracy | Interpretation |
|---------------|----------------|
| ~50% | Domains are indistinguishable (ideal DANN) |
| ~60-70% | Some domain signal remains, but reduced |
| ~90%+ | Domains are easily separable (no adaptation) |

In [ ]:
# The API is straightforward:
# accuracy = domain_confusion_score(
#     model,
#     source_dataset,
#     target_dataset,
#     device='cuda',
#     max_samples=500,
# )

# What it does internally:
# 1. Extract features from source and target datasets
# 2. Train a small linear probe (10 epochs of logistic regression)
# 3. Evaluate on a 30% held-out set
# 4. Return the accuracy

# Simulated results:
print("Example results:")
print(f"  Baseline model:  probe accuracy = 0.94  (domains easily separable)")
print(f"  DANN model:      probe accuracy = 0.56  (domain confusion achieved!)")
print()
print("Interpretation: the DANN encoder has learned features where")
print("source and target are nearly indistinguishable.")

---
## 4. Full Analysis Workflow

Here is the complete workflow for comparing baseline vs. DANN:

```python
import torch
import numpy as np
import yaml

from udm_epic4.models.dann import DANNModel
from udm_epic4.data.multi_domain_dataset import DomainDataset
from udm_epic4.evaluation.domain_analysis import (
    extract_features, compute_tsne, plot_tsne, domain_confusion_score
)

# 1. Load config
with open('configs/epic4_dann.yaml') as f:
    cfg = yaml.safe_load(f)

# 2. Load trained model
model = DANNModel(backbone='convnext_tiny', pretrained=False)
state = torch.load('outputs/epic4_dann/best.pth', map_location='cpu')
model.load_state_dict(state['model_state_dict'])
model.eval()

# 3. Load datasets
source_ds = DomainDataset('data/warstein/images', 'data/warstein/masks', domain_label=0)
target_ds = DomainDataset('data/malaysia/images', domain_label=1)

# 4. Extract features
src_feats, src_labels = extract_features(model, source_ds, device='cpu')
tgt_feats, tgt_labels = extract_features(model, target_ds, device='cpu')

# 5. t-SNE
all_feats = np.concatenate([src_feats, tgt_feats])
all_labels = np.concatenate([src_labels, tgt_labels])
coords = compute_tsne(all_feats)
plot_tsne(coords, all_labels, ['warstein', 'malaysia'],
          save_path='outputs/tsne_dann.png')

# 6. Domain confusion score
acc = domain_confusion_score(model, source_ds, target_ds, device='cpu')
print(f'Domain confusion probe accuracy: {acc:.1%}')
```

---
## Summary

- `extract_features()` gets bottleneck feature vectors from any dataset.
- `compute_tsne()` + `plot_tsne()` visualise domain overlap in 2D.
- `domain_confusion_score()` quantifies domain separability (lower = better).
- Successful DANN: t-SNE shows overlapping clusters, probe accuracy near 50%.

**Next:** `epic4_05_deployment.ipynb` -- evaluate the model on all domains.